In [1]:
######### Plot violin for Landsat signature across three land cover types ########
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

In [11]:
def data_prep(path, file, cutoff=0, direction=0, cutlon=0, sensor = 1):
    ####### filter the data with 015042 index ######
    df = pd.read_csv(path + file)
    df['ID'] = range(0, len(df))  # Generates IDs starting from 1
    
    if sensor == 0:
        training_data = df[['lat', 'lon', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'ndmi', 'ndvi', 'savi', 'descriptio']].to_numpy()
    else:
        training_data = df[['lat', 'lon', 'SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7', 'ndmi', 'ndvi', 'savi', 'descriptio']].to_numpy()
    
    predictor = []
    target = []
    coord = []
    # Loop through each (1585, 10) array in training_data
    print('training data is:')
    #print(training_data)
    print(training_data.shape)
    #for array in training_data:
        # Get rows where the target is not 'TZ'
    mask = training_data[:, 11] != 'MZ'
    
    predictor.append(training_data[mask, 2:11])  # First 9 columns
    target.append(training_data[mask, 11])       # Target column
    coord.append(training_data[mask, :2])
    
    # Convert to NumPy arrays if desired
    predictor = np.array(predictor)
    target = np.array(target)
    coord = np.array(coord)
    
    X = predictor.reshape(-1, 9)  # Shape: (5 * 1585, 9)
    y = target.reshape(-1)        # Shape: (5 * 1585,)
    coord = coord.reshape(-1, 2)
    ### cleanse the data by removing nan ####
    valid_indices = []

    # Loop through each row and check for NaN in the first column
    for i in range(X.shape[0]):
        if not np.isnan(X[i, 0]):
            valid_indices.append(i)
    
    # Create new X and y with only valid rows
    X = X[valid_indices]
    y = y[valid_indices]
    coord = coord[valid_indices]

    print('the size of dataset is:')
    print(X.shape[0])

    if cutoff:
        if direction > 0: ### keep the eastern side
            masklon = coord[:, 1] > cutlon
        else:
            masklon = coord[:, 1] <= cutlon
    
        X = X[masklon]
        y = y[masklon]
        coord = coord[masklon]
    
        print('the size of lon filtered dataset is:')
        print(X.shape[0])
        #print(X)
    
    return X, y, coord

def filter_percentile(df):
    lower = df['Value'].quantile(0.10)
    upper = df['Value'].quantile(0.90)
    return df[(df['Value'] >= lower) & (df['Value'] <= upper)]

In [3]:
X_cal = {}
y_cal = {}
coord_cal = {}

sensorshift1 = 0
calyears = [2023, 2018]

for year in calyears:
    X, y, coord = data_prep(f'/Volumes/Malonelab/Research/Whitezone/Data/KML/{year}/', 'median.csv', 0, 0, 0, sensorshift1)
    X_cal[str(year)] = X.astype(float)
    y_cal[str(year)] = y
    coord_cal[str(year)] = coord

sensorshift2 = 1
calyears = [2013, 2009, 2003, 1999, 1994, 1985]

for year in calyears:
    X, y, coord = data_prep(f'/Volumes/Malonelab/Research/Whitezone/Data/KML/{year}/', 'median.csv', 0, 0, 0, sensorshift2)
    X_cal[str(year)] = X.astype(float)
    y_cal[str(year)] = y
    coord_cal[str(year)] = coord

training data is:
(1583, 12)
the size of dataset is:
1476
training data is:
(1584, 12)
the size of dataset is:
1462
training data is:
(1586, 12)
the size of dataset is:
1455
training data is:
(1584, 12)
the size of dataset is:
1452
training data is:
(1581, 12)
the size of dataset is:
1448
training data is:
(1577, 12)
the size of dataset is:
1444
training data is:
(1619, 12)
the size of dataset is:
1488
training data is:
(1579, 12)
the size of dataset is:
1439


In [18]:
%matplotlib agg
dfs = []
fs = 17
land_cover_order = ['WZ', 'VZ', 'TZ']
filtered_df['Land_Cover'] = pd.Categorical(filtered_df['Land_Cover'], categories=land_cover_order, ordered=True)

for year in X_cal.keys():
    # Swap Predictor 7 and 8
    predictors = [f'Predictor_{i}' for i in range(1, 10)]
    predictors[6], predictors[7] = predictors[7], predictors[6]  # Swap index 6 and 7 (Predictor_7 and Predictor_8)

    X_year = pd.DataFrame(X_cal[year], columns=predictors)
    y_year = pd.Series(y_cal[year], name='Land_Cover')
    df_year = pd.concat([X_year, y_year], axis=1)
    df_year['Year'] = year
    dfs.append(df_year)

X_df_all = pd.concat(dfs, ignore_index=True)

# Normalize predictors 1-6
cols_to_normalize = [f'Predictor_{i}' for i in range(1, 7)]
scaler = MinMaxScaler()
X_df_all[cols_to_normalize] = scaler.fit_transform(X_df_all[cols_to_normalize])

# Melt to long format
melted_df = pd.melt(
    X_df_all,
    id_vars=['Land_Cover', 'Year'],
    value_vars=[f'Predictor_{i}' for i in range(1, 10)],
    var_name='Predictor',
    value_name='Value'
)

# Apply the filtering group-wise by 'Predictor' and 'Land_Cover'
filtered_df = melted_df.groupby(['Predictor', 'Land_Cover'], group_keys=False).apply(filter_percentile)

# Define readable x-axis labels
predictor_labels = {
    'Predictor_1': 'Blue',
    'Predictor_2': 'Green',
    'Predictor_3': 'Red',
    'Predictor_4': 'Near-infrared',
    'Predictor_5': 'Shortwave infrared 1',
    'Predictor_6': 'Shortwave infrared 2',
    'Predictor_7': 'NDVI',  # Swapped
    'Predictor_8': 'NDMI',  # Swapped
    'Predictor_9': 'SAVI'
}
filtered_df['Predictor_Label'] = filtered_df['Predictor'].map(predictor_labels)

# Now plot with the filtered data
plt.figure(figsize=(14, 6))
sns.set(style='whitegrid', font='Times New Roman')

sns.violinplot(
    data=filtered_df,
    x='Predictor_Label',
    y='Value',
    hue='Land_Cover',
    hue_order=land_cover_order,  # Enforces legend order
    inner='quartile',
    palette='Set2',
    split=False
)

plt.xlabel('', fontsize=fs)
plt.ylabel('Value', fontsize=fs)
plt.xticks(rotation=45, fontsize=fs, fontname='Times New Roman')
plt.yticks(fontsize=fs, fontname='Times New Roman')

plt.legend(
    title='Land Cover',
    bbox_to_anchor=(1.05, 1),
    loc='upper left',
    fontsize=fs,
    title_fontsize=fs
)
plt.tight_layout()
plt.show()
plt.savefig("/Users/bz294/Documents/WhiteZone/Figure/land_cover.tiff", format='tiff', dpi=300, bbox_inches='tight')


/var/folders/yw/3l8t5xh50v33m6zjql7k1rzsvhfd38/T/ipykernel_36922/3211460097.py:35: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  filtered_df = melted_df.groupby(['Predictor', 'Land_Cover'], group_keys=False).apply(filter_percentile)
/var/folders/yw/3l8t5xh50v33m6zjql7k1rzsvhfd38/T/ipykernel_36922/3211460097.py:79: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
